In [3]:
import pandas as pd
import numpy as np

In [4]:
df = pd.read_csv(r"WA_Fn-UseC_-Telco-Customer-Churn.csv")
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [5]:
df.info()
# TotalCharges is an (int) but in the data (object).


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [6]:
df["TotalCharges"].sample(n=30)
# This column contains values ​​written as a space, therefore it is considered an object ---> exp ( row numper : 1340 ).


6184    2237.55
1550      457.1
2282    6822.15
2991    3371.75
3194     2342.2
5345     2335.3
4677      990.9
3964     7511.3
4137      232.4
4963    3522.65
4113     695.85
5101     1672.1
1158     5475.9
5759      70.15
6710    3023.65
2226     813.45
1911     7657.4
6033     2291.2
3857      93.55
4935    1174.35
2338    1322.85
4973       1261
6683     1931.3
154       113.5
216      3139.8
4691    5688.05
2846      113.1
20        39.65
64       857.25
1197     340.85
Name: TotalCharges, dtype: object

In [7]:
df.describe()

,SeniorCitizen,tenure,MonthlyCharges
count,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692
std,0.368612,24.559481,30.090047
min,0.000000,0.000000,18.250000
25%,0.000000,9.000000,35.500000
50%,0.000000,29.000000,70.350000
75%,0.000000,55.000000,89.850000
max,1.000000,72.000000,118.750000


In [8]:
TotalCharges_numeric = pd.to_numeric(df["TotalCharges"] , errors="coerce")

In [9]:
TotalCharges_numeric.isnull().sum()

np.int64(11)

In [17]:
from sklearn.model_selection import train_test_split

train_set = df.drop(columns=["Churn"])
test_set = df["Churn"]

x_train , x_test ,y_train , y_test = train_test_split(train_set , test_set , test_size=0.2 , stratify= test_set , random_state=42)

In [20]:
explore_data = train_set.copy()
explore_data['Churn'] = y_train

In [22]:
print(explore_data.groupby("Contract")["Churn"].value_counts(normalize = True))

Contract        Churn
Month-to-month  No       0.572534
                Yes      0.427466
One year        No       0.889173
                Yes      0.110827
Two year        No       0.971302
                Yes      0.028698
Name: proportion, dtype: float64


In [23]:
print(explore_data[explore_data["TotalCharges"].str.strip() == '' ][['tenure', 'MonthlyCharges', 'TotalCharges']])

      tenure  MonthlyCharges TotalCharges
488        0           52.55             
753        0           20.25             
936        0           80.85             
1082       0           25.75             
1340       0           56.05             
3331       0           19.85             
3826       0           25.35             
4380       0           20.00             
5218       0           19.70             
6670       0           73.35             
6754       0           61.90             


In [34]:
total_serveces = ['PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 
            'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']

explore_data["totla_serveces"] = explore_data[total_serveces].apply(lambda row : sum((row != "NO") & (row != "No phone service")) , axis= 1)

In [39]:
from sklearn.base import BaseEstimator , TransformerMixin

class TotalChargesTransformer(BaseEstimator , TransformerMixin):
    def __init__(self):
        pass

    def fit(self , x , y=None):
        return self
    
    def trasform(self , x):
        x_copy = x.copy()

        if "TotalCharges" in x_copy.colomn:
            x_copy["TotalCharges"] = x_copy["TotalCharges"].replace(r'^\s*$' , np.nan , regex=True)
            x_copy["TotalCharges"] = x_copy["TotalCharges"].astype(float)
            x_copy["TotalCharges"] = x_copy["TotalCharges"].fillna(0.0)

        return x_copy
        

In [43]:
class ServiceAggregator (BaseEstimator , TransformerMixin):
    def __init__(self , services_columns):
        self.services_columns = services_columns

    def fit(self , x , y = None):
        return self
    
    def transform (self , x):
        x_copy = x.copy()


        x_copy["total_serveces"] = x_copy[self.services_columns].apply(

            lambda row : sum((row != "NO") & (row != "No phone service" )), axis= 1
        )

        if 'MonthlyCharges' in x_copy.columns :
            x_copy["cost_per_secvice"] = x_copy["MonthlyCharges"] / (x_copy["TotalCharges"] + 0.1)

        return x_copy
        